# Chapter 8: Heteroscedasticity

Topics: heteroscedasticity-robust inference, Breusch-Pagan and White tests, WLS estimation.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('hprice1')

## 8.1 Heteroscedasticity-Robust Inference

## Step 2 – What is Heteroscedasticity?

**Homoscedasticity** (MLR.5): $\text{Var}(u_i|\mathbf{x}_i) = \sigma^2$ for all $i$.

**Heteroscedasticity**: $\text{Var}(u_i|\mathbf{x}_i) = \sigma_i^2$, which varies across observations.

**Consequences:**
- OLS $\hat{\boldsymbol{\beta}}$ remains **unbiased and consistent**
- Standard errors, t-stats, F-stats are **invalid** → wrong inference
- OLS is **no longer efficient** (BLUE fails)

**Detection:** plot residuals $\hat{u}_i$ or $\hat{u}_i^2$ against $\hat{y}_i$ or regressors.

In [ ]:
# OLS with and without robust SEs
m_ols    = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit()
m_robust = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit(cov_type='HC3')

comparison = pd.DataFrame({
    'OLS SE':    m_ols.bse,
    'Robust SE': m_robust.bse,
    'OLS t':     m_ols.tvalues,
    'Robust t':  m_robust.tvalues
})
print("Standard vs Robust Standard Errors:")
print(comparison.round(4))

## 8.2 Heteroscedasticity Tests

## Step 3 – Heteroscedasticity-Robust Standard Errors

The **heteroscedasticity-robust (sandwich) variance estimator**:
$$\widehat{\text{Var}}(\hat{\boldsymbol{\beta}})_{HC} = (\mathbf{X}'\mathbf{X})^{-1}\left(\sum_{i=1}^n \hat{u}_i^2 \mathbf{x}_i\mathbf{x}_i'\right)(\mathbf{X}'\mathbf{X})^{-1}$$

**HC3** (recommended for small samples) uses bias-corrected residuals: $\hat{u}_i / (1 - h_{ii})$ where $h_{ii}$ is the leverage.

In statsmodels: `results = model.fit(cov_type='HC3')`

Robust SEs are valid regardless of the form of heteroscedasticity.

In [ ]:
# Breusch-Pagan test
bp_stat, bp_pval, _, _ = het_breuschpagan(m_ols.resid, m_ols.model.exog)
print(f"Breusch-Pagan test: stat={bp_stat:.4f}, p-value={bp_pval:.4f}")
print("→ Reject H0 of homoscedasticity" if bp_pval < 0.05 else "→ Fail to reject H0")

# White test
w_stat, w_pval, _, _ = het_white(m_ols.resid, m_ols.model.exog)
print(f"\nWhite test: stat={w_stat:.4f}, p-value={w_pval:.4f}")

# Residual plot
plt.figure(figsize=(8, 4))
plt.scatter(m_ols.fittedvalues, m_ols.resid, alpha=0.5, color='steelblue')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values'); plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.savefig(f"{output_dir}/fig1_residual_plot.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_08_Heteroscedasticity_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8.3 Weighted Least Squares (WLS)

## Step 4 – Testing for Heteroscedasticity

**Breusch-Pagan test:**
1. Regress $\hat{u}_i^2$ on all regressors $\mathbf{x}_i$
2. LM statistic: $LM = n \cdot R^2_{\hat{u}^2} \sim \chi^2_k$ under H₀

**White test:** same as BP but also includes squares and cross-products of regressors.

H₀: homoscedasticity. Reject if p-value $< \alpha$.

**WLS** (Weighted Least Squares) – if the form of heteroscedasticity is known:
$$\min_{\boldsymbol{\beta}} \sum_{i=1}^n w_i(y_i - \mathbf{x}_i'\boldsymbol{\beta})^2, \quad w_i = 1/\sigma_i^2$$

In statsmodels: `smf.wls(formula, data=df, weights=1/sigma2)`

In [ ]:
# Weight by 1/lotsize (proxy for variance)
m_wls = smf.wls('price ~ lotsize + sqrft + bdrms', data=df,
                  weights=1/df['lotsize']).fit()

print("WLS Results:")
print(m_wls.summary())

# Compare OLS vs WLS coefficients
comp = pd.DataFrame({'OLS': m_ols.params, 'WLS': m_wls.params})
print("\nOLS vs WLS Coefficients:")
print(comp.round(4))